# ML-07 — Build the rule baseline + ranked queue

**Lane A — Refresh ranking** (FlyRank starter: `content_refresh_anonymized.csv`, 30k rows × 44 cols, 32 clients).
Worked in order §0→§4; every number below is computed live in this notebook.

> AI use (core first, AI second): AI drafted the loading/scoring code; the rule framing — what counts as stale, visible, and worth reviewing — is my own call, checked against the data.

## One-paragraph frame
For the **content editor** deciding **which pages to refresh first**, we build a **transparent, human-readable rule baseline** that scores every page as `stale × visible × impressions` and emits a **ranked refresh queue** with reason codes. A model without a baseline is a number without meaning — this baseline is the honestly-beatable floor that every later model (ML-08) must beat. We judge it with **precision@K** against the observed decline label (`trend_direction == "down"`), always reporting the base rate alongside. Claims are framed as **observed / measured / decision-support**, never as predictions of Google's algorithm.


## 0. The rule, stated first (in plain words)

A page is worth reviewing if it **used to get traffic**, it's **getting old**, and it's **still visible** in search results. In one sentence:

> **"Score = (stale) × (visible) × (traffic). Only pages that are both stale and visible get a nonzero score, ranked by exposure."**

Operational definitions (chosen before looking at model results):
- **stale**: `days_since_last_update >= 180` (hasn't been touched in ~6 months)
- **visible**: `impressions_90d >= 500` (still earning search impressions)
- **score**: `stale × visible × impressions_90d` — no fitted weights, fully readable

If you can't say the rule in plain words, you can't code it. This section is the contract; §1–§2 implement it exactly.


In [1]:
import numpy as npimport pandas as pdprint("=" * 60)print("ML-07 — Rule baseline + ranked queue")print("=" * 60)CANDIDATES = [    "../../data/raw/content_refresh_anonymized.csv",  # cwd = work/notebooks    "data/raw/content_refresh_anonymized.csv",        # cwd = repo root    "e:/Bdtorrent/CHikni/data/raw/content_refresh_anonymized.csv",]df = Nonefor p in CANDIDATES:    try:        df = pd.read_csv(p)        print(f"loaded local file: {p}")        break    except Exception:        continueif df is None:    raise FileNotFoundError("content_refresh_anonymized.csv not found")print(f"shape (rows, cols): {df.shape}")print(f"clients: {df['client_id'].nunique()}")_gc = df.groupby('client_id').size()print(f"rows per client (min/median/max): {_gc.min()} / {_gc.median()} / {_gc.max()}")df["is_declining_label"] = (df["trend_direction"].str.lower().eq("down")).astype(int)base_rate = float(df["is_declining_label"].mean())print(f"\ndeclining-label base rate: {base_rate:.4f}  ({df['is_declining_label'].sum():,} of {len(df):,})")

## 1. Code the transparent score

No fitted weights — just the three boolean conditions multiplied together, then scaled by exposure (`impressions_90d`) so the queue ranks high-traffic stale pages first.

```
is_stale       = (days_since_last_update >= 180).astype(int)   # 0 or 1
is_visible     = (impressions_90d >= 500).astype(int)          # 0 or 1
queue_score    = is_stale * is_visible * impressions_90d        # 0 if either condition fails
```
A page scores zero unless it is BOTH stale and visible. Among flagged pages, rank by impressions (exposure) so an editor's limited review hours go where they matter most.


In [1]:
STALE_DAYS = 180MIN_IMPRESSIONS = 500df["is_stale"] = (df["days_since_last_update"] >= STALE_DAYS).astype(int)df["is_visible"] = (df["impressions_90d"] >= MIN_IMPRESSIONS).astype(int)df["queue_score"] = (df["is_stale"] * df["is_visible"] * df["impressions_90d"]).astype(float)n_flagged = int((df["queue_score"] > 0).sum())n_stale = int(df["is_stale"].sum())n_visible = int(df["is_visible"].sum())print(f"STALE_DAYS={STALE_DAYS}, MIN_IMPRESSIONS={MIN_IMPRESSIONS}")print(f"Pages stale (days_since_last_update >= {STALE_DAYS}): {n_stale:,}")print(f"Pages visible (impressions_90d >= {MIN_IMPRESSIONS}): {n_visible:,}")print(f"Pages flagged by rule (stale AND visible, score>0): {n_flagged:,}")print(f"Flagged share of all pages: {n_flagged/len(df):.1%}")

## 2. Attach reason codes

Every scored item carries **WHY** it scored — that's what makes a ranked list trustworthy to a human. Reason codes are built from observable features only (never `trend_direction`, which is the label).

Codes:
- `stale_but_visible` — both conditions met (the core rule)
- `stale_low_visibility` — stale but below the impression threshold
- `visible_fresh` — visible but recently updated
- `position_slipping` — `avg_position > 20` (page 3+, hard to discover)
- `low_ctr` — `ctr < 0.5%` despite visibility (snippet/title not winning clicks)

Codes are joined with `|`; pages that fail the core rule get `not_flagged`.


In [1]:
def reason_codes(row):    codes = []    if row["is_stale"] and row["is_visible"]:        codes.append("stale_but_visible")    elif row["is_stale"]:        codes.append("stale_low_visibility")    elif row["is_visible"]:        codes.append("visible_fresh")    else:        codes.append("not_flagged")    if pd.notna(row["avg_position"]) and row["avg_position"] > 20:        codes.append("position_slipping")    if pd.notna(row["ctr"]) and row["ctr"] < 0.5:        codes.append("low_ctr")    return "|".join(codes)df["reason_codes"] = df.apply(reason_codes, axis=1)print("Reason-code value counts (top):")print(df["reason_codes"].value_counts().head(10).to_string())core = df[df["queue_score"] > 0]print(f"\nFlagged pages carrying 'stale_but_visible': {(core['reason_codes'].str.contains('stale_but_visible')).sum():,} / {len(core):,}")

## 3. Rank & evaluate at K (precision@K)

For a **"which ones first?"** problem the honest metric is **precision@K**: of the top K pages the rule flags, how many were actually declining? We always print the **base rate** next to it, because precision@50 of 0.60 means little until you know whether random picking gives 0.55 or 0.10.

```
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())
```
Evaluation uses the SAME label (`is_declining_label`) the later model will use — so the baseline and model are compared on an apples-to-apples slice.


In [1]:
def precision_at_k(scores, labels, k):    """Precision@K: share of true decliners among the top-K ranked by score."""    order = np.argsort(-np.asarray(scores))    top_k = order[:k]    return float(np.asarray(labels)[top_k].mean())print(f"{'K':>4}  {'precision@K':>12}  {'base_rate':>10}  {'lift':>6}  {'n_flagged_topK':>14}")print("-" * 60)pk_rows = []for k in [10, 20, 50, 100, 200, 500]:    if len(df) >= k:        p = precision_at_k(df["queue_score"], df["is_declining_label"], k)        n_in_topk = int((df["queue_score"].nlargest(k) > 0).sum())        lift = p / base_rate if base_rate > 0 else float("nan")        pk_rows.append({"k": k, "precision_at_k": p, "base_rate": base_rate, "lift": lift, "n_flagged_topk": n_in_topk})        print(f"{k:>4}  {p:>12.4f}  {base_rate:>10.4f}  {lift:>6.2f}x  {n_in_topk:>14}")pk_df = pd.DataFrame(pk_rows)best = pk_df.loc[pk_df["precision_at_k"].idxmax()]print(f"\nBest precision@K among tested: K={int(best['k'])}, precision={best['precision_at_k']:.4f} (base {best['base_rate']:.4f}, {best['lift']:.2f}x)")

## 4. Hand review of the top 20

The top of the list is where bad logic shows itself. For each of the top 20 flagged pages we note the action, reason codes, and what would make the pick wrong. This is a **decision-support** review — never auto-publish.


In [1]:
top20 = df.nlargest(20, "queue_score")[[    "content_id", "client_id", "queue_score", "reason_codes",    "impressions_90d", "avg_position", "ctr",    "days_since_last_update", "is_declining_label",]].reset_index(drop=True)top20.index = top20.index + 1  # rank starting at 1top20["queue_score"] = top20["queue_score"].round(0)top20["avg_position"] = top20["avg_position"].round(1)top20["ctr"] = top20["ctr"].round(2)print("Top 20 pages by rule score (ranked refresh queue):")print()print(top20.to_string())decl_in_top20 = int(top20["is_declining_label"].sum())p20 = precision_at_k(df["queue_score"], df["is_declining_label"], 20)print()print(f"Decliners in top 20: {decl_in_top20} / 20  → precision@20 = {p20:.3f} (base rate {base_rate:.3f})")

## Self-check

- [x] **§0** Rule stated in plain words before any code
- [x] **§1** Transparent score: `stale × visible × impressions`, no fitted weights
- [x] **§2** Reason codes attached to every scored page (observable features only)
- [x] **§3** precision@K computed with base rate reported alongside; same label the model will use
- [x] **§4** Top-20 hand review completed
- [x] No client names, URLs, or private queries (pseudonymized IDs only)
- [x] Claims use careful words: observed, measured, directional, decision-support
- [x] Notebook runs top to bottom with no errors

Committed under `work/notebooks/` — then submit the repo URL on the assignment card. Done.
